In [0]:
storage_account = "deproject01"
container = "retail-data"

base_path = (
    f"abfss://{container}@"
    f"{storage_account}.dfs.core.windows.net"
)

display(dbutils.fs.ls(f"{base_path}/source/"))

path,name,size,modificationTime
abfss://retail-data@deproject01.dfs.core.windows.net/source/customer/,customer/,0,1792398922000
abfss://retail-data@deproject01.dfs.core.windows.net/source/product/,product/,0,1792398928000
abfss://retail-data@deproject01.dfs.core.windows.net/source/sales/,sales/,0,1792398907000
abfss://retail-data@deproject01.dfs.core.windows.net/source/stores/,stores/,0,1792398939000


In [0]:
paths = {
    "sales": f"{base_path}/source/sales/sales.csv",
    "customers": f"{base_path}/source/customers/customers.csv",
    "products": f"{base_path}/source/products/products.csv",
    "stores": f"{base_path}/source/stores/stores.csv"
}

for name, path in paths.items():
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(path)
    )

    print(f"\n{name.upper()}: {df.count():,} rows")
    print(df.columns)
    df.show(3, truncate=False)


SALES: 100,000 rows
['order_id', 'customer_id', 'product_id', 'store_id', 'order_ts', 'order_date', 'order_month', 'country', 'status', 'quantity', 'amount']
+--------+-----------+----------+--------+-------------------+----------+-----------+-------+---------+--------+------+
|order_id|customer_id|product_id|store_id|order_ts           |order_date|order_month|country|status   |quantity|amount|
+--------+-----------+----------+--------+-------------------+----------+-----------+-------+---------+--------+------+
|0       |1971       |218       |31      |2024-04-17 17:53:03|2024-04-17|2024-04-01 |IN     |SHIPPED  |3       |68.89 |
|1       |4819       |451       |42      |2023-02-26 12:32:46|2023-02-26|2023-02-01 |DE     |COMPLETED|4       |321.48|
|2       |1316       |192       |26      |2023-06-12 02:20:03|2023-06-12|2023-06-01 |IN     |COMPLETED|3       |134.39|
+--------+-----------+----------+--------+-------------------+----------+-----------+-------+---------+--------+------+
o

In [0]:
# Show the current catalog and schema
print("Current catalog and schema:")
spark.sql("""
    SELECT current_catalog(), current_schema()
""").show(truncate=False)

# List available catalogs
print("Available catalogs:")
spark.sql("SHOW CATALOGS").show(truncate=False)

In [0]:
spark.sql("SHOW TABLES IN de_catalog.bronze").show(truncate=False)

In [0]:
customers_path = (
    "abfss://retail-data@deproject01.dfs.core.windows.net/"
    "source/customers/customers.csv"
)

customers_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(customers_path)
)

print("Customer row count:", customers_df.count())
print("Customer columns:", customers_df.columns)

customers_df.printSchema()
display(customers_df.limit(5))

In [0]:
# Save the customers DataFrame as a managed Delta table
(
    customers_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("de_catalog.bronze.customers_raw")
)

print("Customers Bronze table created successfully.")

In [0]:
spark.sql(
    "SHOW TABLES IN de_catalog.bronze"
).show(truncate=False)

saved_customers = spark.table(
    "de_catalog.bronze.customers_raw"
)

print("Saved customer rows:", saved_customers.count())
display(saved_customers.limit(5))

In [0]:
stores_path = (
    "abfss://retail-data@deproject01.dfs.core.windows.net/"
    "source/stores/stores.csv"
)

stores_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(stores_path)
)

print("stores row count:", stores_df.count())
print("stores columns:", stores_df.columns)

customers_df.printSchema()
display(stores_df.limit(5))

In [0]:
# Save the customers DataFrame as a managed Delta table
(
    stores_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("de_catalog.bronze.stores_raw")
)

print("stores Bronze table created successfully.")

In [0]:
products_path = (
    "abfss://retail-data@deproject01.dfs.core.windows.net/"
    "source/products/products.csv"
)

products_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(products_path)
)

print("products row count:", stores_df.count())
print("products columns:", stores_df.columns)

products_df.printSchema()
display(products_df.limit(5))

In [0]:
# Save the customers DataFrame as a managed Delta table
(
    products_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("de_catalog.bronze.products_raw")
)

print("products Bronze table created successfully.")

In [0]:
sales_path = (
    "abfss://retail-data@deproject01.dfs.core.windows.net/"
    "source/sales/sales.csv"
)

sales_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(sales_path)
)

print("Sales row count:", sales_df.count())
sales_df.printSchema()
display(sales_df.limit(5))

In [0]:
(
    sales_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("de_catalog.bronze.sales_raw")
)

print("Sales Bronze table created successfully.")